In [133]:
import operator
import os
import re
from typing import Annotated, List, Literal

import wikipedia
from ddgs import DDGS
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Send

load_dotenv()  

True

In [134]:
import faiss

In [135]:
from langchain_community.document_loaders import PyPDFLoader

pdfs = [
    r"D:\d\NLP\corrective_RAG\1706.03762v7.pdf",
    r"D:\d\NLP\corrective_RAG\2005.11401v4.pdf",
    r"D:\d\NLP\corrective_RAG\2210.03629v3.pdf",
    r"D:\d\NLP\corrective_RAG\2310.11511v1.pdf",
    r"D:\d\NLP\corrective_RAG\2401.15884v3.pdf",
]
whole_pdfs =[]
for pdf in pdfs:
    load = PyPDFLoader(pdf).load()
    whole_pdfs.extend(load)
whole_pdfs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\d\\NLP\\corrective_RAG\\1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz

In [136]:
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings()
splitter = RecursiveCharacterTextSplitter(chunk_size = 500 , chunk_overlap = 100)
chunks = splitter.split_documents(whole_pdfs)
vector_store = FAISS.from_documents(chunks, embeddings)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 299.23it/s]


In [137]:
vs=vector_store.as_retriever()

In [138]:
from langchain_core.documents import Document


class GraphState(BaseModel):
    question: str
    messages: Annotated[List[AnyMessage], add_messages] = Field(default_factory=list)
    documents: List[Document] = Field(default_factory=list)
    feedback: str = ""
    generation: str = ""



In [139]:
### Retrieve node
def retrieve_node(state:GraphState)->dict:
    results = vs.invoke(state.question)
    return {"documents" : results}



In [140]:
retrieve_node(GraphState(question="what is the transformers architecture about"))


{'documents': [Document(id='fe7835dc-19f1-4a2c-890f-89d53c540e0f', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\d\\NLP\\corrective_RAG\\1706.03762v7.pdf', 'total_pages': 15, 'page': 2, 'page_label': '3'}, page_content='Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The first is a multi-head self-attention mechanism, and the sec

In [141]:
from langchain_groq import ChatGroq
load_dotenv()
llm = ChatGroq(model="qwen/qwen3.8-27b")
llm.invoke("wassup")


AIMessage(content="Not much, just here and ready to help! What's on your mind today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 14, 'total_tokens': 32, 'completion_time': 0.035705758, 'completion_tokens_details': None, 'prompt_time': 0.000834637, 'prompt_tokens_details': None, 'queue_time': 0.018567494, 'total_time': 0.036540395}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_43dd1ea4ee', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a12dbd-84fb-7a31-928e-4a52df2e17de-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 18, 'total_tokens': 32})

In [142]:
from langchain_ollama import ChatOllama
llm2 = ChatOllama(model="qwen3:14b")
llm2.invoke("wassup").content

"Hey there! 😊 What's up? Hope you're doing well! Let me know if there's anything I can help with."

In [143]:
GRADER_PROMPT = SystemMessage(content="""You are a relevance grader in a Corrective RAG pipeline.
You will receive a user QUESTION and one retrieved DOCUMENT chunk from a research paper.

Decide whether the document contains information that helps answer the question.

Grade "Yes" if the chunk:
- directly answers the question, or part of it, or
- contains facts, definitions, methods, or results that are clearly about the question's topic.

Grade "No" if the chunk:
- is only loosely related (shares a keyword but not the meaning),
- is a reference list, author list, acknowledgements, table of contents, or other boilerplate,
- is about a different concept than the one asked about.

Be strict: the goal is to filter out irrelevant chunks, not to answer the question.
Judge only the given chunk; do not use outside knowledge to fill gaps.""")


In [144]:
class GradeDocument(BaseModel):
    """Relevance grade for one retrieved chunk."""
    relevant: Literal["Yes", "No"] = Field(description="'Yes' if the chunk is relevant to the question, else 'No'")

grader = llm.with_structured_output(GradeDocument)

def grade_document(state: GraphState) -> dict:
    relevant_docs = []
    for doc in state.documents:
        result = grader.invoke([
            GRADER_PROMPT,
            HumanMessage(content=f"QUESTION:\n{state.question}\n\nDOCUMENT:\n{doc.page_content}"),
        ])
        if result.relevant == "Yes":
            relevant_docs.append(doc)

    return {
        "documents": relevant_docs,
        "feedback": "Yes" if relevant_docs else "No",
    }


In [145]:
REWRITE_PROMPT = SystemMessage(content="""You are a query rewriter in a Corrective RAG pipeline.
The user's QUESTION could not be answered well from the local document store, so it will be sent to a web search engine.

Rewrite the question into a single, concise web search query that is most likely to return relevant, factual results.

Guidelines:
- Keep the original intent and all key entities, names, acronyms, and technical terms.
- Expand ambiguous acronyms when it helps (e.g. "RAG" -> "retrieval-augmented generation").
- Remove filler words, greetings, and conversational phrasing.
- Prefer keywords and specific terms over full sentences; about 5-15 words.
- Do not answer the question, add facts, or introduce assumptions not present in it.
- Output only the rewritten query, with no explanation or quotes.""")


In [146]:
class RewrittenQuery(BaseModel):
    """Web-search-optimized version of the user's question."""
    query: str = Field(description="The rewritten web search query")

rewriter = llm.with_structured_output(RewrittenQuery)

def transform_query_node(state: GraphState) -> dict:
    result = rewriter.invoke([
        REWRITE_PROMPT,
        HumanMessage(content=f"QUESTION:\n{state.question}"),
    ])
    return {"question": result.query}


In [147]:
## web search node
def web_search_node(state: GraphState) -> dict:
    try:
        results = DDGS().text(state.question, max_results=5)
    except Exception as e:
        print(f"Web search failed: {e}")
        return {}  

    web_docs = [
        Document(page_content=r["body"], metadata={"source": r["href"], "title": r["title"]})
        for r in results
    ]
    return {"documents": state.documents + web_docs}


In [148]:
GENERATION_PROMPT = SystemMessage(content="""You are a research assistant answering questions about machine learning papers.
You will receive a user QUESTION and a set of CONTEXT passages. Each passage is either a chunk from a research paper or a web search result, and each one is labeled with its source.

Answer the question using only the information in the context.

Guidelines:
- Base every claim on the context; do not add facts from your own knowledge.
- If the context only partly answers the question, answer that part and clearly say what is missing.
- If the context does not contain the answer, say "I could not find this in the provided sources." and do not guess.
- Cite the source of each key claim in square brackets, e.g. [1706.03762v7.pdf, p. 3] or [https://example.com].
- If sources disagree, point out the disagreement instead of picking one silently.
- Be clear and concise: start with a direct answer, then add supporting detail. Use short paragraphs or bullet points when that is easier to read.
- Explain technical terms briefly when they are central to the answer.""")


In [149]:
### generate node

def format_docs(docs: List[Document]) -> str:
    parts = []
    for doc in docs:
        source = os.path.basename(doc.metadata.get("source", "unknown"))
        page = doc.metadata.get("page")
        label = f"{source}, p. {page + 1}" if page is not None else source
        parts.append(f"[{label}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


def generate_node(state: GraphState) -> dict:
    context = format_docs(state.documents)
    response = llm.invoke([
        GENERATION_PROMPT,
        HumanMessage(content=f"QUESTION:\n{state.question}\n\nCONTEXT:\n{context}"),
    ])
    return {"generation": response.content, "messages": [response]}


In [150]:
def router_node(state:GraphState)-> Literal["transform_query","generate"]:
    if state.feedback =="Yes":
        return "generate"
    return "transform_query"
    

In [151]:
graph = StateGraph(GraphState)
graph.add_node("retrieve",retrieve_node)
graph.add_node("grade_documents",grade_document)
graph.add_node("transform_query",transform_query_node)
graph.add_node("web_search",web_search_node)
graph.add_node("generate",generate_node)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve","grade_documents")
graph.add_conditional_edges("grade_documents",router_node)
graph.add_edge("grade_documents","transform_query")
graph.add_edge("transform_query","web_search")
graph.add_edge("web_search","generate")
graph.add_edge("generate",END)

graph_builder = graph.compile()




In [152]:
test_cases = [
    # In the PDFs: should go retrieve -> grade -> generate
    {"question": "Why is dot-product attention scaled by 1/sqrt(d_k) in the Transformer?", "expect": "local"},       # 1706.03762 Attention
    {"question": "How does RAG combine parametric and non-parametric memory?", "expect": "local"},                  # 2005.11401 RAG
    {"question": "How does ReAct interleave reasoning traces with actions?", "expect": "local"},                     # 2210.03629 ReAct
    {"question": "What are reflection tokens in Self-RAG and what are they used for?", "expect": "local"}
]

for case in test_cases:
    print("=" * 80)
    print("Q:", case["question"], f"(expected: {case['expect']})")
    path = []
    for update in graph_builder.stream({"question": case["question"]}, stream_mode="updates"):
        path.extend(update.keys())
        if "generate" in update:
            final = update["generate"]
    print("Path:", " -> ".join(path))
    print("Went to web:", "web_search" in path)
    print("Answer:", final["generation"][:500], "...")


Q: Why is dot-product attention scaled by 1/sqrt(d_k) in the Transformer? (expected: local)
Path: retrieve -> grade_documents -> transform_query -> generate -> web_search -> generate
Went to web: True
Answer: The 1/√d_k scaling factor in Transformer scaled dot-product attention is used to prevent the dot products from becoming too large in magnitude when the key dimension (d_k) is large. Without this scaling, large dot product values push the softmax function into regions where it has extremely small gradients, which can hinder training. Scaling by 1/√d_k counteracts this effect by normalizing the variance of the dot products, keeping the softmax inputs within a range that allows for stable and effec ...
Q: How does RAG combine parametric and non-parametric memory? (expected: local)
Path: retrieve -> grade_documents -> generate -> transform_query -> web_search -> generate
Went to web: True
Answer: RAG (Retrieval-Augmented Generation) models combine **parametric memory** and **non-param